Input Test Cases for Optimisation:

In [2]:
import pandas as pd

sheets = ['employees', 'vehicles', 'baseline', 'metadata']
def load_excel(filename):
    testcase = dict()
    try:
        for sheet in sheets:
            testcase[sheet] = pd.read_excel(filename, sheet_name=sheet)
        print(f"Input {filename} Successful.")
        return testcase
    except Exception as e:
        print(f"Error: {e}")
        return None

Loading Transportation Map of Bengaluru:

In [3]:
import osmium as osm
import numpy as np

In [3]:
osmfile = 'GeoData/bengaluru.osm'

nodes = {}
xnodes = {}
r_nodes = {}
edges = {}
r_edges = {}
ways = {}
ls_lat = []
ls_lng = []

R = 6731
def haversine_distance(nd1, nd2):
    dlat = np.radians(abs(nodes[nd1][0] - nodes[nd2][0]))
    dlon = np.radians(abs(nodes[nd1][1] - nodes[nd2][1]))
    d = 2 * R * np.arcsin(np.sqrt((np.sin(dlat / 2)**2 + np.cos(np.radians(nodes[nd1][0])) * np.cos(np.radians(nodes[nd2][0])) * np.sin(dlon / 2)**2)))
    return d

In [6]:
class OSMHandler(osm.SimpleHandler):
    def node(self, n):
        nodes[n.id] = (n.location.lat, n.location.lon)
        xnodes[(n.location.lat, n.location.lon)] = n.id
        edges[n.id] = []
        r_edges[n.id] = []

    def way(self, w):
        ways[w.id] = w.tags
        nds = w.nodes

        road = False
        oneway = False
        access = True
        for t in w.tags:
            if t.k == 'highway':
                road = True
            if t.k == 'oneway' and t.v == 'yes':
                oneway = True
                break
            if t.k == 'access' and (t.k == 'no' or t.k == 'private'):
                access = False
            if t.k == 'motor_vehicle' and t.v == 'no':
                access = False
        if road and access:
            for n in nds:
                r_nodes[n.ref] = nodes[n.ref]
                ls_lat.append(nodes[n.ref][0])
                ls_lng.append(nodes[n.ref][1])
            ls_lat.append(None)
            ls_lng.append(None)
            
            nc = len(nds)
            for i in range(1, nc):
                d = haversine_distance(nds[i - 1].ref, nds[i].ref)
                edges[nds[i - 1].ref].append((nds[i].ref, d))
                if not oneway:
                    edges[nds[i].ref].append((nds[i - 1].ref, d))
                r_edges[nds[i].ref].append((nds[i - 1].ref, d))
                if not oneway:
                    r_edges[nds[i - 1].ref].append((nds[i].ref, d))

handler = OSMHandler()
handler.apply_file(osmfile)

In [14]:
import csv
node_file = 'graph_nodes.csv'
edge_file = 'graph_edges.csv'

def load_nodes(file):
    nf = open(file, 'r')
    nr = csv.DictReader(nf)
    for row in nr:
        row['id'] = int(row['id'])
        row['lat'] = float(row['lat'])
        row['lng'] = float(row['lng'])
        nodes[row['id']] = (row['lat'], row['lng'])
        r_nodes[row['id']] = (row['lat'], row['lng'])
        xnodes[(row['lat'], row['lng'])] = row['id']
        edges[row['id']] = []
        r_edges[row['id']] = []
    nf.close()

def load_edges(file):
    ef = open(file, 'r')
    er = csv.DictReader(ef)
    for row in er:
        row['id1'] = int(row['id1'])
        row['id2'] = int(row['id2'])
        row['length'] = float(row['length'])
        edges[row['id1']].append((row['id2'], row['length']))
        r_edges[row['id2']].append((row['id1'], row['length']))
    ef.close()

In [17]:
load_nodes(node_file)
load_edges(edge_file)

In [16]:
print(len(nodes))
print(len(r_nodes))

1867063
1867063


Manage using Neo4j:

In [7]:
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = "bhowal12345" 

In [30]:
import csv
def create_CSV():
    f_nodes = open('graph_nodes.csv', 'w')
    w_nodes = csv.writer(f_nodes)
    w_nodes.writerow(['id', 'lat', 'lng'])
    dat = []
    for n in r_nodes:
        dat.append([n, r_nodes[n][0], r_nodes[n][1]])
    w_nodes.writerows(dat)
    f_nodes.close()
    print('Nodes CSV file Created!')
    f_edges = open('graph_edges.csv', 'w')
    w_edges = csv.writer(f_edges)
    w_edges.writerow(['id1', 'id2', 'length'])
    dat = []
    for e1 in edges:
        for e2, d in edges[e1]:
            dat.append([e1, e2, d])
    w_edges.writerows(dat)
    f_edges.close()
    print('Edges CSV file Created!')

def upload_graph():
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    driver.verify_connectivity()
    query2 = '''
    UNWIND $data as edge
    MERGE (a:node {id: edge.id1, lat: edge.lat1, lng: edge.lng1})
    MERGE (b:node {id: edge.id2, lat: edge.lat2, lng: edge.lng2})
    MERGE (a)-[:way {id: edge.id, length: edge.length}]->(b)
    '''
    COUNT = 10000
    dat = []
    w_id = 1
    for e1 in edges:
        for e2, d in edges[e1]:
            dat.append({'id': w_id, 'id1': e1, 'id2': e2, 'lat1': r_nodes[e1][0], 'lat2': r_nodes[e2][0], 'lng1': r_nodes[e1][1], 'lng2': r_nodes[e2][1], 'length': d})
            w_id += 1
    for i in range(0, len(dat), COUNT):
        summary = driver.execute_query(query2, data = dat[i: min(len(dat), i + COUNT)], database_="neo4j").summary
        print(f"Created: {summary.counters.nodes_created} nodes and {summary.counters.relationships_created} ways in {summary.result_available_after} ms.")

    driver.close()
    print('Graph uploaded!')

In [29]:
upload_graph()

Created: 1294 nodes and 1000 ways in 2648 ms.
Created: 1068 nodes and 1000 ways in 606 ms.
Created: 1251 nodes and 1000 ways in 666 ms.
Created: 998 nodes and 1000 ways in 822 ms.
Created: 1164 nodes and 1000 ways in 425 ms.
Created: 1167 nodes and 1000 ways in 165 ms.
Created: 894 nodes and 1000 ways in 136 ms.
Created: 1151 nodes and 1000 ways in 206 ms.
Created: 1054 nodes and 1000 ways in 126 ms.
Created: 1189 nodes and 1000 ways in 113 ms.
Created: 988 nodes and 1000 ways in 202 ms.
Created: 760 nodes and 1000 ways in 124 ms.
Created: 500 nodes and 1000 ways in 119 ms.
Created: 1224 nodes and 1000 ways in 315 ms.
Created: 1287 nodes and 1000 ways in 188 ms.
Created: 1184 nodes and 1000 ways in 150 ms.
Created: 1231 nodes and 1000 ways in 137 ms.
Created: 869 nodes and 1000 ways in 190 ms.
Created: 1061 nodes and 1000 ways in 119 ms.
Created: 861 nodes and 1000 ways in 114 ms.
Created: 968 nodes and 1000 ways in 109 ms.
Created: 930 nodes and 1000 ways in 94 ms.
Created: 890 nodes 

Pre Computations:

In [63]:
import heapq
import sys
from scipy.spatial import KDTree

points = np.array(list(r_nodes.values()))
tree = KDTree(points)

Methods:

In [74]:
def nearest_node(loc):
    md, p = tree.query(loc, k=1)
    p = tuple(points[p])
    nrst = xnodes[p]
    #print(f'Node {nrst} found {md * 10**3} m away')
    return nrst

def dijkstras(dest):
    pq = []
    dist = {}
    for n in r_nodes.keys():
        dist[n] = sys.maxsize
    dist[dest] = 0
    heapq.heappush(pq, (0, dest))

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        
        for v, w in r_edges[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(pq, (dist[v], v))
    return dist

def reconstruct_path(came_from, cur):
    path = [cur]
    while cur in came_from:
        cur = came_from[cur]
        path.append(cur)
    return path[::-1]

def astar(src, dest):
    open_set = []
    heapq.heappush(open_set, (0, src))

    came_from = {} # path reconstruction
    g_dist = {src: 0}
    f_dist = {src: haversine_distance(src, dest)}

    while open_set:
        _, cur = heapq.heappop(open_set)
        if cur == dest:
            return reconstruct_path(came_from, cur), g_dist[cur]
        for ngh, dist in edges[cur]:
            tg = g_dist[cur] + dist
            if ngh not in g_dist or tg < g_dist[ngh]:
                came_from[ngh] = cur
                g_dist[ngh] = tg
                f_dist[ngh] = tg + haversine_distance(ngh, dest)
                heapq.heappush(open_set, (f_dist[ngh], ngh))
    return None, float('inf')

def optimal_route(src, dest):
    route, length = astar(src, dest)
    #print('heuristic_route:', route)
    #print('Length:', length)
    route_lat = []
    route_lng = []
    if route is not None:
        for n in route:
            route_lat.append(r_nodes[n][0])
            route_lng.append(r_nodes[n][1])
        route_lat.append(None)
        route_lng.append(None)
    return length, route


Applying on TestCases:

In [84]:
from datetime import time
def to_mins(hrs):
    return 60 * hrs.hour + hrs.minute 

def to_hrs(mins):
    return time(hour = mins // 60, minute = mins % 60)


In [ ]:
def solve(file):
    tc = load_excel(file)
    employees_df, vehicles_df, baseline_df, metadata_df = tc['employees'], tc['vehicles'], tc['baseline'], tc['metadata']

    tc_id = metadata_df.iat[0, 1]
    print(f'ID: {tc_id}')

    drop = nearest_node((employees_df.iat[0, 4], employees_df.iat[0, 5]))
    employees = {}
    vehicles = {}
    metadata = {}
    considered_nodes = [drop]

    for kv in metadata_df.itertuples():
        metadata[kv.key] = kv.value

    for vh in vehicles_df.itertuples():
        depot = nearest_node((vh.current_lat, vh.current_lng))
        vehicles[vh.vehicle_id] = {'capacity': vh.capacity, 'cost/km': vh.cost_per_km, 'speed': vh.avg_speed_kmph, 'depot': depot, 'avail_from': to_mins(vh.available_from), 'premium': (vh.category == 'premium')}
        considered_nodes.append(depot)

    for emp in employees_df.itertuples():
        pick = nearest_node((emp.pickup_lat, emp.pickup_lng))
        delay = metadata['priority_' + str(emp.priority) + '_max_delay_min']
        employees[emp.employee_id] = {'pick': pick, 'sharing': emp.sharing_preference, 'time_window': (to_mins(emp.earliest_pickup), to_mins(emp.latest_drop) + delay)}    
        considered_nodes.append(pick)

    cost_weight = metadata['objective_cost_weight']
    time_weight = metadata['objective_time_weight']
    
    # n x n matrix of dist and routes
    dist = {}
    for n1 in considered_nodes:
        dist[n1] = {}
        for n2 in considered_nodes:
            if n1 != n2:
                dist[n1][n2] = optimal_route(n1, n2)
            else:
                dist[n1][n2] = 0, [n1]
        print(n1)

    print("N x N matric created!")

    #solution
    def intial_solution():
        soln = {}
        return soln
    def objective_func(solution):
        cost = 0
        time = 0
        for v_id in solution:
            s = vehicles[v_id]['depot']
            for e_id in solution[v_id]:
                d = employees[e_id]['pick']
                cost += dist[s][d][0] * vehicles[v_id]['cost/km']
                time += dist[s][d][0] / vehicles[v_id]['speed']
                s = d
            d = drop
            cost += dist[s][d][0] * vehicles[v_id]['cost/km']
            time += dist[s][d][0] / vehicles[v_id]['speed']
        return cost_weight * cost + time_weight * time
    def all_neighbours(solution):
        neighbour_list = []
        #remove nodes
        removal_list = []
        
        #insert nodes

        return neighbour_list
    
    #Tabu Search
    current_sol = dict(intial_solution())
    best_sol = dict(current_sol)
    best_func = objective_func(best_sol)
    tabu_list = []
    iterations = 100
    tabu_size = 10

    for _ in range(iterations):
        neighbors = all_neighbours(current_sol)
        obj_func = objective_func(current_sol)
        
        # Sort neighbors by objective function
        neighbors.sort(key=lambda x: objective_func(x[0]))
        
        for neighbor in neighbors:
            n_func = objective_func(neighbor)
            if n_func < obj_func:
                current_sol = neighbor
                obj_func = n_func

        tabu_list.append(current_sol)
        if len(tabu_list) > tabu_size:
            tabu_list.pop(0)
        
        if obj_func < best_func:
            best_sol = dict(current_sol)
            best_func = obj_func

    print("Tabu Search Complete!")
    #upload answer to neo4j
    answer = []
    for vh in best_sol:
        s = vehicles[vh]['depot']
        x = vh
        for e_id in best_sol[vh]:
            d = employees[e_id]['pick']
            answer.append({'name1': x, 'name2': e_id, 'id1': s, 'id2': d, 'lat1': r_nodes[s][0], 'lat2': r_nodes[d][0], 'lng1': r_nodes[s][1], 'lng2': r_nodes[d][1], 'length': dist[s][d][0], 'route': dist[s][d][1]})
            s = d
            x = e_id
        answer.append({'name1': x, 'name2': 'Office', 'id1': s, 'id2': d, 'lat1': r_nodes[s][0], 'lat2': r_nodes[d][0], 'lng1': r_nodes[s][1], 'lng2': r_nodes[d][1], 'length': dist[s][d][0], 'route': dist[s][d][1]})
    upload_graph(answer)
    print("Neo4j Upload Complete!")

    return employees, vehicles, best_sol

In [96]:
solve('TestCases/TestCase_TC02.xlsx')

Input TestCases/TestCase_TC02.xlsx Successful.
ID: TC_02
10086861529
363182695
13074488707
12093301924
7331215175
10091498099
9930197132
1610035079
10863982907
13392488507
308875370
3740962149
10839118540
7263195953
308875893
7086325431
1700826065
7467098491


[]

Handling Requests and Responses:

In [70]:
from flask import *

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'

@app.route('/input')
def input():
    return render_template('input.html')

@app.route('/output', methods = ['GET', 'POST'])
def output():
    if request.method == 'POST':
        if 'file' not in request.files:
            return "No file part", 400
        file = request.files['file']
        if file.filename == '':
            return "No selected file", 400
        if file and file.filename.endswith(('.xls', '.xlsx')):
            res = solve(file)
            return render_template('output.html', result = res)
        else:
            return "Invalid file format. Please upload an Excel file.", 400

    return render_template('input.html')

if __name__ == '__main__':
    app.run(debug = True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1